In [ ]:
# Google Trends, HCUP, & U.S. Census Bureau Data Collection  
**Author:** J. Casey Brookshier  
**Date:** July 15, 2026  

## Objective

The objective of this notebook is to collect, clean, and prepare datasets required to predict suicide related state-level inpatient 
psychiatric admissions using Google Trends search activity.

This notebook will:

1. **Collect Google Trends Data (Leading Indicators)**
   - Retrieve state-level suicide- and crisis-related search interest data (2013–2023/2024).
   - Extract terms such as:
     - "suicidal thoughts"
     - "suicide hotline"
     - "self harm"
     - "suicide prevention"
   - Convert monthly search interest into state-year predictive features.

2. **Collect HCUP Data (Healthcare Utilization Outcome)**
   - Retrieve state-level Mental Health/Substance Use inpatient admission data.
   - Aggregate quarterly utilization measures into annual state-level totals.
   - Create the prediction target:

     Admission Rate = (Mental Health/Substance Use inpatient stays / State population) × 100,000


3. **Collect U.S. Census Bureau Data (Population Controls)**
   - Retrieve annual state population estimates (2013–2023/2024).
   - Use population estimates to calculate per-capita inpatient admission rates and normalize comparisons across states.

## Data Integration Workflow

The final state-year panel will combine:

- **Google Trends:** Leading indicators of potential future psychiatric demand.
- **HCUP:** Actual inpatient psychiatric utilization (prediction target).
- **U.S. Census Bureau:** Population normalization factors.

Workflow:

**Collect → Clean → Transform → Merge → Explore → Model → Visualize → Interpret**

The final dataset will support regression, Random Forest, XGBoost, PCA, hierarchical clustering, and SHAP-based feature importance analysis to evaluate whether Google Trends improves one-year prediction of state-level inpatient psychiatric admissions.


In [1]:
import os

print(os.getcwd())
print(os.path.exists(
    "/Users/caseybrookshier/Desktop/Data Science job attainment planning/DS Portfolio github friendly/Team-Rho-Capstone-Project.cb/data/raw"
))

/Users/caseybrookshier
True


In [2]:
# !pip install urllib3==1.26.18
# !pip install pyreadr
# !pip install pyreadstat

In [5]:
"""
Google Trends Data Collection for Team Rho Capstone Project
-----------------------------------------------------------
Purpose:
Create state-year Google Trends predictors (2013-2023) for forecasting
future state-level suicide related inpatient psychiatric admissions.

"""

from pytrends.request import TrendReq
import pandas as pd
import time
import os
import random


# ----------------------------------------------------
# Connection
# ----------------------------------------------------

pytrends = TrendReq(
    hl="en-US",
    tz=360
)


# ----------------------------------------------------
# Search terms
# ----------------------------------------------------

SEARCH_TERMS = [
    "suicidal thoughts",
    "suicide hotline",
    "self harm",
    "mental health crisis",
    "suicide prevention",
    "crisis hotline",
    "psychiatric hospital",
    "depression help"
]


# ----------------------------------------------------
# States
# ----------------------------------------------------

STATES = {
    "AL":"Alabama",
    "AK":"Alaska",
    "AZ":"Arizona",
    "AR":"Arkansas",
    "CA":"California",
    "CO":"Colorado",
    "CT":"Connecticut",
    "DE":"Delaware",
    "FL":"Florida",
    "GA":"Georgia",
    "HI":"Hawaii",
    "ID":"Idaho",
    "IL":"Illinois",
    "IN":"Indiana",
    "IA":"Iowa",
    "KS":"Kansas",
    "KY":"Kentucky",
    "LA":"Louisiana",
    "ME":"Maine",
    "MD":"Maryland",
    "MA":"Massachusetts",
    "MI":"Michigan",
    "MN":"Minnesota",
    "MS":"Mississippi",
    "MO":"Missouri",
    "MT":"Montana",
    "NE":"Nebraska",
    "NV":"Nevada",
    "NH":"New Hampshire",
    "NJ":"New Jersey",
    "NM":"New Mexico",
    "NY":"New York",
    "NC":"North Carolina",
    "ND":"North Dakota",
    "OH":"Ohio",
    "OK":"Oklahoma",
    "OR":"Oregon",
    "PA":"Pennsylvania",
    "RI":"Rhode Island",
    "SC":"South Carolina",
    "SD":"South Dakota",
    "TN":"Tennessee",
    "TX":"Texas",
    "UT":"Utah",
    "VT":"Vermont",
    "VA":"Virginia",
    "WA":"Washington",
    "WV":"West Virginia",
    "WI":"Wisconsin",
    "WY":"Wyoming",
    "DC":"District of Columbia"
}


# ----------------------------------------------------
# Files
# ----------------------------------------------------

RAW_FILE = "google_trends_raw_progress.csv"

FINAL_FILE = "google_trends_state_year_2013_2023.csv"

FAILED_FILE = "google_trends_failed_queries.csv"


TIMEFRAME = "2013-01-01 2023-12-31"

MAX_RETRIES = 5


# ----------------------------------------------------
# Load existing progress
# ----------------------------------------------------

if os.path.exists(RAW_FILE):

    raw = pd.read_csv(RAW_FILE)

    records = [raw]

    completed = set(
        zip(
            raw["State"],
            raw["Search_Term"]
        )
    )

else:

    records = []

    completed = set()


failed_queries = []


# ----------------------------------------------------
# Retrieval function
# ----------------------------------------------------

def get_google_trends(term, geo):

    for attempt in range(MAX_RETRIES):

        try:

            pytrends.build_payload(
                kw_list=[term],
                timeframe=TIMEFRAME,
                geo=geo
            )

            data = pytrends.interest_over_time()

            if not data.empty:

                return data


        except Exception as e:

            wait = 10 * (2 ** attempt)

            print(
                f"Retry {attempt+1}/{MAX_RETRIES}; waiting {wait}s"
            )

            time.sleep(wait)


    return None



# ----------------------------------------------------
# Download
# ----------------------------------------------------

for state_abbrev, state_name in STATES.items():

    geo = f"US-{state_abbrev}"

    print(
        f"\nProcessing {state_name}"
    )


    for term in SEARCH_TERMS:


        if (state_name, term) in completed:

            print(
                f"Skipping {state_name} - {term}"
            )

            continue


        trends = get_google_trends(
            term,
            geo
        )


        if trends is None:

            failed_queries.append(
                {
                    "State": state_name,
                    "Term": term
                }
            )

            continue


        trends = trends.reset_index()

        trends["Year"] = (
            trends["date"]
            .dt.year
        )


        annual = (
            trends
            .groupby("Year")[term]
            .mean()
            .reset_index()
        )


        annual["State"] = state_name

        annual["Search_Term"] = term


        records.append(
            annual
        )


        # Save checkpoint immediately
        checkpoint = pd.concat(
            records,
            ignore_index=True
        )


        checkpoint.to_csv(
            RAW_FILE,
            index=False
        )


        time.sleep(
            random.uniform(5,10)
        )


# ----------------------------------------------------
# Create state-year panel
# ----------------------------------------------------

raw = pd.concat(
    records,
    ignore_index=True
)


panel = (
    raw
    .pivot_table(
        index=[
            "State",
            "Year"
        ],
        columns="Search_Term",
        values=raw.columns[1],
        aggfunc="mean"
    )
    .reset_index()
)


panel.columns = [
    str(c)
    .replace(" ","_")
    .replace("-","_")
    .lower()
    for c in panel.columns
]


# ----------------------------------------------------
# Add lag variables
# ----------------------------------------------------

panel = panel.sort_values(
    [
        "state",
        "year"
    ]
)


predictors = [
    c for c in panel.columns
    if c not in [
        "state",
        "year"
    ]
]


for col in predictors:

    panel[col + "_lag1"] = (
        panel
        .groupby("state")[col]
        .shift(1)
    )


# ----------------------------------------------------
# Save final ML dataset
# ----------------------------------------------------

panel.to_csv(
    FINAL_FILE,
    index=False
)


if failed_queries:

    pd.DataFrame(
        failed_queries
    ).to_csv(
        FAILED_FILE,
        index=False
    )


print("COMPLETE")
print(panel.head())


Processing Alabama
Retry 1/5; waiting 10s
Retry 2/5; waiting 20s
Retry 3/5; waiting 40s

Processing Alaska

Processing Arizona

Processing Arkansas

Processing California
Retry 1/5; waiting 10s

Processing Colorado

Processing Connecticut
Retry 1/5; waiting 10s

Processing Delaware

Processing Florida

Processing Georgia

Processing Hawaii

Processing Idaho

Processing Illinois

Processing Indiana

Processing Iowa
Retry 1/5; waiting 10s
Retry 1/5; waiting 10s

Processing Kansas

Processing Kentucky

Processing Louisiana

Processing Maine

Processing Maryland

Processing Massachusetts

Processing Michigan

Processing Minnesota

Processing Mississippi

Processing Missouri

Processing Montana

Processing Nebraska
Retry 1/5; waiting 10s

Processing Nevada

Processing New Hampshire
Retry 1/5; waiting 10s

Processing New Jersey

Processing New Mexico

Processing New York

Processing North Carolina

Processing North Dakota

Processing Ohio

Processing Oklahoma

Processing Oregon

Processing 

In [6]:
# Copy files to project folder
import os
import shutil

# ----------------------------------------------------
# Source files created by Google Trends notebook
# ----------------------------------------------------

files_to_copy = [
    "google_trends_raw_progress.csv",
    "google_trends_state_year_2013_2023.csv",
    "google_trends_failed_queries.csv"
]


# ----------------------------------------------------
# Destination folder
# ----------------------------------------------------

destination = r"/Users/caseybrookshier/Desktop/Data Science job attainment planning/DS Portfolio github friendly/Team-Rho-Capstone-Project.cb/data/raw"


# Create destination if it does not exist

os.makedirs(
    destination,
    exist_ok=True
)


# ----------------------------------------------------
# Copy files
# ----------------------------------------------------

for file in files_to_copy:

    if os.path.exists(file):

        shutil.copy(
            file,
            destination
        )

        print(
            f"Copied: {file}"
        )

    else:

        print(
            f"Not found (skipped): {file}"
        )


print("\nCSV transfer complete.")

Copied: google_trends_raw_progress.csv
Copied: google_trends_state_year_2013_2023.csv
Copied: google_trends_failed_queries.csv

CSV transfer complete.


In [7]:
#View data set head/dimensions
import pandas as pd

# Load raw Google Trends progress file
raw_trends = pd.read_csv(
    "google_trends_raw_progress.csv"
)

# Display first five rows
raw_trends.head()

# View dimensions
print("Shape:", raw_trends.shape)

# View column names
print("\nColumns:")
print(raw_trends.columns.tolist())

# Display first rows
raw_trends.head()

Shape: (4411, 11)

Columns:
['Year', 'suicidal thoughts', 'State', 'Search_Term', 'suicide hotline', 'self harm', 'mental health crisis', 'suicide prevention', 'crisis hotline', 'psychiatric hospital', 'depression help']


,Year,suicidal thoughts,State,Search_Term,suicide hotline,self harm,mental health crisis,suicide prevention,crisis hotline,psychiatric hospital,depression help
0,2013,66.833333,Alabama,suicidal thoughts,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014,42.083333,Alabama,suicidal thoughts,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,47.250000,Alabama,suicidal thoughts,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016,52.250000,Alabama,suicidal thoughts,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017,50.833333,Alabama,suicidal thoughts,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:

# ==========================================================
# Build State Population Dataset (2013-2023)
# Uses:
#   nst-est2020.xlsx
#   NST-EST2025-POP.xlsx
# ==========================================================

import pandas as pd
from pathlib import Path

# ----------------------------------------------------------
# File locations
# ----------------------------------------------------------

DATA_DIR = Path("/Users/caseybrookshier/Desktop/Data Science job attainment planning/DS Portfolio github friendly/Team-Rho-Capstone-Project.cb/data/raw")

OLD_FILE = DATA_DIR / "nst-est2020.xlsx"
NEW_FILE = DATA_DIR / "NST-EST2025-POP.xlsx"

OUTPUT_FILE = DATA_DIR / "uscb_state_population_2013_2023.csv"


# ----------------------------------------------------------
# Read entire worksheets (no header)
# ----------------------------------------------------------

old_raw = pd.read_excel(OLD_FILE, header=None)
new_raw = pd.read_excel(NEW_FILE, header=None)


# ----------------------------------------------------------
# Locate first state (.Alabama)
# ----------------------------------------------------------

old_start = old_raw[old_raw.iloc[:,0].astype(str).str.startswith(".Alabama")].index[0]
new_start = new_raw[new_raw.iloc[:,0].astype(str).str.startswith(".Alabama")].index[0]


# ----------------------------------------------------------
# Build OLD (2013-2020)
# ----------------------------------------------------------

old_years = [
    "State","2010","Base","2010","2011","2012",
    "2013","2014","2015","2016","2017",
    "2018","2019","2020_Apr","2020"
]

old = old_raw.iloc[old_start:].copy()
old.columns = old_years

old = old[
    ["State","2013","2014","2015","2016",
     "2017","2018","2019","2020"]
]


# ----------------------------------------------------------
# Build NEW (2021-2023)
# ----------------------------------------------------------

new_years = [
    "State","Base","2020","2021",
    "2022","2023","2024","2025"
]

new = new_raw.iloc[new_start:].copy()
new.columns = new_years

new = new[
    ["State","2021","2022","2023"]
]


# ----------------------------------------------------------
# Merge
# ----------------------------------------------------------

wide = old.merge(new,on="State",how="left")

# Remove leading dots from state names
wide["State"] = (
    wide["State"]
    .astype(str)
    .str.replace(".","",regex=False)
    .str.strip()
)

# Keep only states + DC
exclude = {
    "United States",
    "Northeast",
    "Midwest",
    "South",
    "West",
    "Puerto Rico"
}

wide = wide[~wide["State"].isin(exclude)]


# ----------------------------------------------------------
# Convert to long format
# ----------------------------------------------------------

population = wide.melt(
    id_vars="State",
    var_name="Year",
    value_name="state_population"
)

population["Year"] = population["Year"].astype(int)
population["state_population"] = (
    pd.to_numeric(population["state_population"], errors="coerce")
    .astype("Int64")
)

population = (
    population
    .sort_values(["State","Year"])
    .reset_index(drop=True)
)

population.to_csv(OUTPUT_FILE,index=False)

print("Saved:", OUTPUT_FILE)
print("Rows:", len(population))
print(population.head(20))

Saved: /Users/caseybrookshier/Desktop/Data Science job attainment planning/DS Portfolio github friendly/Team-Rho-Capstone-Project.cb/data/raw/uscb_state_population_2013_2023.csv
Rows: 638
      State  Year  state_population
0   Alabama  2013           4831586
1   Alabama  2014           4843737
2   Alabama  2015           4854803
3   Alabama  2016           4866824
4   Alabama  2017           4877989
5   Alabama  2018           4891628
6   Alabama  2019           4907965
7   Alabama  2020           4921532
8   Alabama  2021           5050058
9   Alabama  2022           5076868
10  Alabama  2023           5117850
11   Alaska  2013            737626
12   Alaska  2014            737075
13   Alaska  2015            738430
14   Alaska  2016            742575
15   Alaska  2017            740983
16   Alaska  2018            736624
17   Alaska  2019            733603
18   Alaska  2020            731158
19   Alaska  2021            734590


In [16]:
# Inspect Structure of HCUP data tab

import pandas as pd
from pathlib import Path

RAW = Path(
    "/Users/caseybrookshier/Desktop/Data Science job attainment planning/"
    "DS Portfolio github friendly/Team-Rho-Capstone-Project.cb/data/raw"
)

HCUP_FILE = RAW / "DownloadTable_StatePayerIP_2026-03-20.xls"

# Read the actual data worksheet
df = pd.read_excel(HCUP_FILE, sheet_name="Data")

print(df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 25 rows:")
print(df.head(25))

print("\nData types:")
print(df.dtypes)

(1935, 94)

Columns:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'HCUP Fast Stats State-Level Trends in Inpatient Stays by Payer and Year, SID and Quarterly Data', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Un

In [19]:
# ==========================================================
# HCUP CLEANING PIPELINE
#
# Input:
#   DownloadTable_StatePayerIP_2026-03-20.xls
#
# Output:
#   hcup_state_year_inpatient_admissions_2013_2023.csv
#
# Purpose:
#   Create merge-ready State x Year inpatient admissions table
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

RAW = Path(
    "/Users/caseybrookshier/Desktop/Data Science job attainment planning/"
    "DS Portfolio github friendly/Team-Rho-Capstone-Project.cb/data/raw"
)

HCUP_FILE = RAW / "DownloadTable_StatePayerIP_2026-03-20.xls"

OUTPUT = RAW / "hcup_state_year_inpatient_admissions_2013_2023.csv"


# ----------------------------------------------------------
# Read HCUP Data sheet WITHOUT header
# ----------------------------------------------------------

raw = pd.read_excel(
    HCUP_FILE,
    sheet_name="Data",
    header=None
)


print("Raw HCUP shape:")
print(raw.shape)


# ----------------------------------------------------------
# Locate actual header row
# ----------------------------------------------------------

header_candidates = []

for i in range(len(raw)):

    row = raw.iloc[i].astype(str).tolist()

    if (
        "State" in row
        and
        "Hospitalization Type" in row
        and
        "Expected Payer" in row
    ):
        header_candidates.append(i)


print("\nHeader candidates:")
print(header_candidates)


if len(header_candidates) == 0:

    raise ValueError(
        "Could not locate HCUP header row."
    )


header_row = header_candidates[0]


print("\nUsing header row:")
print(header_row)



# ----------------------------------------------------------
# Reload using correct header
# ----------------------------------------------------------

hcup = pd.read_excel(
    HCUP_FILE,
    sheet_name="Data",
    header=header_row
)



# Clean columns

hcup.columns = (
    hcup.columns
    .astype(str)
    .str.strip()
)


print("\nHCUP columns:")
print(hcup.columns.tolist())



# ----------------------------------------------------------
# Identify quarterly columns
# ----------------------------------------------------------

quarter_cols = [
    c for c in hcup.columns
    if (
        isinstance(c,str)
        and "Q" in c
        and c[:4].isdigit()
    )
]


print("\nQuarter columns:")
print(len(quarter_cols))
print(quarter_cols[:5])
print(quarter_cols[-5:])



# ----------------------------------------------------------
# Convert quarter values to numeric
# ----------------------------------------------------------

for c in quarter_cols:

    hcup[c] = (
        hcup[c]
        .astype(str)
        .str.replace(",","",regex=False)
        .replace(
            ["nan","None",""],
            np.nan
        )
    )

    hcup[c] = pd.to_numeric(
        hcup[c],
        errors="coerce"
    )



# ----------------------------------------------------------
# Keep required years only
# ----------------------------------------------------------

analysis_quarters = [
    c for c in quarter_cols
    if any(
        str(y) in c
        for y in range(2013,2024)
    )
]


hcup = hcup[
    [
        "State",
        "Hospitalization Type",
        "Expected Payer"
    ]
    +
    analysis_quarters
]


# ----------------------------------------------------------
# Aggregate across:
#   payer
#   hospitalization type
#
# Creates total inpatient stays
# ----------------------------------------------------------

hcup_state_quarter = (
    hcup
    .groupby("State")[analysis_quarters]
    .sum()
    .reset_index()
)



# ----------------------------------------------------------
# Quarter -> Annual
# ----------------------------------------------------------

annual = []


for year in range(2013,2024):

    cols = [
        c for c in analysis_quarters
        if str(year) in c
    ]


    temp = hcup_state_quarter[
        ["State"] + cols
    ].copy()


    temp["Year"] = year


    temp["annual_inpatient_stays"] = (
        temp[cols]
        .sum(axis=1)
    )


    annual.append(
        temp[
            [
                "State",
                "Year",
                "annual_inpatient_stays"
            ]
        ]
    )


hcup_clean = pd.concat(
    annual,
    ignore_index=True
)



# ----------------------------------------------------------
# Validation
# ----------------------------------------------------------

print("\nClean HCUP output:")
print(hcup_clean.head(20))


print("\nShape:")
print(hcup_clean.shape)


print("\nStates:")
print(hcup_clean["State"].nunique())


print("\nYears:")
print(hcup_clean["Year"].unique())



# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

hcup_clean.to_csv(
    OUTPUT,
    index=False
)


print("\nSaved:")
print(OUTPUT)

Raw HCUP shape:
(1936, 94)

Header candidates:
[3]

Using header row:
3

HCUP columns:
['State', 'Hospitalization Type', 'Expected Payer', '2003 Q1', '2003 Q2', '2003 Q3', '2003 Q4', '2004 Q1', '2004 Q2', '2004 Q3', '2004 Q4', '2005 Q1', '2005 Q2', '2005 Q3', '2005 Q4', '2006 Q1', '2006 Q2', '2006 Q3', '2006 Q4', '2007 Q1', '2007 Q2', '2007 Q3', '2007 Q4', '2008 Q1', '2008 Q2', '2008 Q3', '2008 Q4', '2009 Q1', '2009 Q2', '2009 Q3', '2009 Q4', '2010 Q1', '2010 Q2', '2010 Q3', '2010 Q4', '2011 Q1', '2011 Q2', '2011 Q3', '2011 Q4', '2012 Q1', '2012 Q2', '2012 Q3', '2012 Q4', '2013 Q1', '2013 Q2', '2013 Q3', '2013 Q4', '2014 Q1', '2014 Q2', '2014 Q3', '2014 Q4', '2015 Q1', '2015 Q2', '2015 Q3', '2015 Q4', '2016 Q1', '2016 Q2', '2016 Q3', '2016 Q4', '2017 Q1', '2017 Q2', '2017 Q3', '2017 Q4', '2018 Q1', '2018 Q2', '2018 Q3', '2018 Q4', '2019 Q1', '2019 Q2', '2019 Q3', '2019 Q4', '2020 Q1', '2020 Q2', '2020 Q3', '2020 Q4', '2021 Q1', '2021 Q2', '2021 Q3', '2021 Q4', '2022 Q1', '2022 Q2', '20

In [ ]:
# ==========================================================
# Dataset Schema Validation and Merge Preparation
#
# PURPOSE
# -------
# Inspect source datasets before building the final
# state-year analytical panel.
#
# Sources:
#   • Google Trends search activity
#   • HCUP inpatient admissions data
#   • U.S. Census population estimates
#
# Output:
#   • Dataset dimensions
#   • Column structures
#   • Data types
#   • Missingness
#   • Candidate merge keys
#   • Shared schema elements
# ==========================================================

import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 100)


# ----------------------------------------------------------
# Project directory
# ----------------------------------------------------------

RAW = Path(
    "/Users/caseybrookshier/Desktop/Data Science job attainment planning/"
    "DS Portfolio github friendly/Team-Rho-Capstone-Project.cb/data/raw"
)

GT_FILE = RAW / "google_trends_raw_progress.csv"
HCUP_FILE = RAW / "DownloadTable_StatePayerIP_2026-03-20.xls"
CENSUS_FILE = RAW / "uscb_state_population_2013_2023.csv"


# ----------------------------------------------------------
# Dataset inspection function
# ----------------------------------------------------------

def inspect(df, name):

    print("\n")
    print("=" * 100)
    print(name.upper())
    print("=" * 100)

    print("\nShape")
    print(df.shape)

    print("\nColumns")
    print(df.columns.tolist())

    print("\nData Types")
    print(df.dtypes)

    print("\nFirst 20 Rows")
    print(df.head(20))

    print("\nLast 10 Rows")
    print(df.tail(10))

    print("\nMissing Values")
    print(df.isna().sum())

    print("\nUnique Values")
    print(df.nunique())

    print("\nCandidate Merge Columns")

    for col in df.columns:

        check = col.lower()

        if any(
            term in check
            for term in [
                "state",
                "year",
                "quarter",
                "month",
                "date",
                "region"
            ]
        ):

            print(f"\n{col}")

            try:
                print(
                    df[col]
                    .drop_duplicates()
                    .head(20)
                    .tolist()
                )
            except Exception:
                pass


# ----------------------------------------------------------
# Google Trends dataset
# ----------------------------------------------------------

google = pd.read_csv(
    GT_FILE
)

inspect(
    google,
    "Google Trends"
)


# ----------------------------------------------------------
# HCUP dataset
# ----------------------------------------------------------

print("\n")
print("=" * 100)
print("HCUP WORKBOOK")
print("=" * 100)


xls = pd.ExcelFile(
    HCUP_FILE
)

print("\nAvailable Sheets")
print(xls.sheet_names)


hcup = pd.read_excel(
    HCUP_FILE
)

inspect(
    hcup,
    "HCUP"
)


# ----------------------------------------------------------
# Census population dataset
# ----------------------------------------------------------

census = pd.read_csv(
    CENSUS_FILE
)

inspect(
    census,
    "U.S. Census Population"
)


# ----------------------------------------------------------
# Schema comparison
# ----------------------------------------------------------

print("\n")
print("=" * 100)
print("COMMON COLUMN NAMES")
print("=" * 100)


common_columns = (
    set(google.columns)
    &
    set(hcup.columns)
    &
    set(census.columns)
)

print(
    sorted(common_columns)
)


print("\n")
print("=" * 100)
print("DATASET VALIDATION COMPLETE")
print("=" * 100)


print(
"""
Inspection complete.

Review output for:

1. Google Trends
   - State/year structure
   - Search term fields
   - Missing values
   - Annual aggregation requirements

2. HCUP
   - Header location
   - State identifier
   - Quarterly admission fields
   - Payer aggregation requirements

3. Census
   - State/year population structure
   - Merge compatibility

Validated datasets can be used to create:

- Annual state-level Google Trends features
- Annual HCUP inpatient admission totals
- Population-adjusted admission rates
- One-year lagged predictor variables
- Final state-year modeling panel

Target modeling applications:

- Fixed-effects regression
- Random Forest
- XGBoost
- PCA
- Hierarchical clustering
- SHAP feature importance
"""
)

In [20]:
# ==========================================================
# TEAM RHO FINAL MERGE PIPELINE
#
# Inputs:
#   google_trends_raw_progress.csv
#   hcup_state_year_inpatient_admissions_2013_2023.csv
#   uscb_state_population_2013_2023.csv
#
# Output:
#   team_rho_state_year_prediction_dataset.csv
#
# Model-ready:
#   State x Year panel
#   Lagged Google predictors
#   HCUP inpatient admissions outcome
# ==========================================================

import pandas as pd
from pathlib import Path


# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

RAW = Path(
    "/Users/caseybrookshier/Desktop/Data Science job attainment planning/"
    "DS Portfolio github friendly/Team-Rho-Capstone-Project.cb/data/raw"
)


GT_FILE = RAW / "google_trends_raw_progress.csv"

HCUP_FILE = RAW / (
    "hcup_state_year_inpatient_admissions_2013_2023.csv"
)

CENSUS_FILE = RAW / (
    "uscb_state_population_2013_2023.csv"
)


OUTPUT = RAW / (
    "team_rho_state_year_prediction_dataset.csv"
)



# ==========================================================
# 1. GOOGLE TRENDS
# ==========================================================

google = pd.read_csv(GT_FILE)


search_terms = [
    "suicidal thoughts",
    "suicide hotline",
    "self harm",
    "mental health crisis",
    "suicide prevention",
    "crisis hotline",
    "psychiatric hospital",
    "depression help"
]


# Fill sparse columns

for term in search_terms:
    google[term] = google[term].fillna(0)



# Collapse to one row per State-Year

google_year = (
    google
    .groupby(
        ["State","Year"]
    )[search_terms]
    .sum()
    .reset_index()
)



# ==========================================================
# 2. HCUP
# ==========================================================

hcup = pd.read_csv(
    HCUP_FILE
)


hcup["State"] = (
    hcup["State"]
    .astype(str)
    .str.strip()
)



# ==========================================================
# 3. CENSUS
# ==========================================================

census = pd.read_csv(
    CENSUS_FILE
)


# Remove Census artifacts

census = census[
    census["State"].notna()
]


census = census[
    census["state_population"].notna()
]


census["State"] = (
    census["State"]
    .astype(str)
    .str.strip()
)


census = census[
    census["Year"].between(2013,2023)
]



# ==========================================================
# 4. MERGE THREE DATASETS
# ==========================================================

dataset = (
    google_year
    .merge(
        hcup,
        on=["State","Year"],
        how="inner"
    )
    .merge(
        census,
        on=["State","Year"],
        how="inner"
    )
)



# ==========================================================
# 5. CREATE ADMISSION RATE
# ==========================================================

dataset["admission_rate_per_100k"] = (
    dataset["annual_inpatient_stays"]
    /
    dataset["state_population"]
    *
    100000
)



# ==========================================================
# 6. CREATE ONE YEAR GOOGLE LAGS
#
# Google Year t --> Admissions Year t+1
# ==========================================================

dataset = (
    dataset
    .sort_values(
        [
            "State",
            "Year"
        ]
    )
)


for term in search_terms:

    dataset[f"{term}_lag1"] = (
        dataset
        .groupby("State")[term]
        .shift(1)
    )



# Remove years without prior Google data

dataset = dataset[
    dataset["Year"] >= 2014
]



# ==========================================================
# 7. Final model dataset
# ==========================================================

final_columns = [
    "State",
    "Year",
    "state_population",
    "annual_inpatient_stays",
    "admission_rate_per_100k"
] + [
    f"{x}_lag1"
    for x in search_terms
]


dataset = dataset[
    final_columns
]



# ==========================================================
# 8. Validation
# ==========================================================

print("="*70)
print("FINAL PROJECT DATASET")
print("="*70)

print(dataset.head())

print("\nShape:")
print(dataset.shape)

print("\nStates:")
print(dataset["State"].nunique())

print("\nYears:")
print(dataset["Year"].unique())

print("\nMissing Values:")
print(dataset.isna().sum())

print("\nDuplicate State-Year Rows:")
print(
    dataset.duplicated(
        ["State","Year"]
    ).sum()
)



# ==========================================================
# 9. Save
# ==========================================================

dataset.to_csv(
    OUTPUT,
    index=False
)


print("\nSaved:")
print(OUTPUT)

FINAL PROJECT DATASET
    State  Year  state_population  annual_inpatient_stays  admission_rate_per_100k  suicidal thoughts_lag1  suicide hotline_lag1  self harm_lag1  mental health crisis_lag1  suicide prevention_lag1  crisis hotline_lag1  psychiatric hospital_lag1  \
1  Alaska  2014          737075.0                     0.0                 0.000000               16.250000              0.000000       15.166667                        0.0                22.083333                  0.0                        0.0   
2  Alaska  2015          738430.0                261050.0             35352.030660                0.000000              8.083333        0.000000                        0.0                12.416667                  0.0                        0.0   
3  Alaska  2016          742575.0                271000.0             36494.630172                0.000000              0.000000       16.000000                        0.0                 9.250000                  0.0                 